# Main — XLM-RoBERTa fine-tune (full)

## Imports and data

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import confusion_matrix, mean_absolute_error

MODEL_NAME  = 'xlm-roberta-base'
MAX_LENGTH  = 256
BATCH_SIZE  = 8
EVAL_BATCH  = 32
GRAD_ACCUM  = 2
LR          = 2e-5
EPOCHS      = 3
WARMUP      = 0.06
WD          = 0.01
SEED        = 42
OUTPUT_DIR  = 'runs/xlmr_full'

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

## Reuse the canonical val split

In [3]:
val_idx = np.load('data/val_indices.npy')
train_df = train.drop(index=val_idx).reset_index(drop=True)
val_df   = train.loc[val_idx].reset_index(drop=True)

## Tokenize

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LENGTH)
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return item

train_ds = ReviewDataset(train_df['sentence'], train_df['label'])
val_ds   = ReviewDataset(val_df['sentence'],   val_df['label'])
test_ds  = ReviewDataset(test['sentence'])

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

## Training setup

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    mae = mean_absolute_error(labels, preds)
    return {'score': 1 - mae / 4, 'mae': mae}

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WD,
    warmup_ratio=WARMUP,
    fp16=False,
    bf16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='score',
    greater_is_better=True,
    seed=SEED,
    report_to=[],
    logging_steps=200,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

## Train

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## Evaluate with MAE-optimal rounding

In [ ]:
def median_round(probs):
    cum = np.cumsum(probs, axis=1)
    return np.argmax(cum >= 0.5, axis=1)

def report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    score = 1 - mae / 4
    acc = float(np.mean(y_true == y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    print(f'[{name}] score={score:.4f}  MAE={mae:.4f}  acc={acc:.4f}')
    print(cm)

val_out = trainer.predict(val_ds)
val_probs = torch.softmax(torch.tensor(val_out.predictions), dim=-1).numpy()

report(val_df['label'].values, np.argmax(val_probs, axis=-1), 'val_argmax')
report(val_df['label'].values, median_round(val_probs),       'val_median')

## Predict and write submission

In [ ]:
os.makedirs('submissions', exist_ok=True)
test_out = trainer.predict(test_ds)
test_probs = torch.softmax(torch.tensor(test_out.predictions), dim=-1).numpy()
test_pred = median_round(test_probs)

pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)}).to_csv(
    'submissions/main_xlmr.csv', index=False,
)